# Phase 5 --- Regime-Conditional Factor Analysis

**Objective:** Quantify how factor behaviour changes across macroeconomic regimes.  
This is the **statistical core** of the factor-timing thesis.

**Inputs:**
- `factor_returns_full.parquet` (hml, umd, rmw, lowvol)
- `regime_probabilities.parquet` (p_expansion, p_slowdown, p_crisis, regime_label)

**Outputs:**
- `outputs/tables/regime_conditional_stats.csv`
- Figures: regime Sharpe heatmap, box plots, correlation heatmaps
- Summary of hypothesis test results

**Key hypotheses:**
- H1: Momentum crashes in crisis-to-recovery transitions
- H2: Value outperforms in early expansion
- H3: Quality/LowVol dominate in crisis
- H4: Cross-factor correlations increase in crisis

---
## 5.1 Setup & Load Dependencies

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import warnings
import logging
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Project imports
from src.config import (
    PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, LOGS_DIR,
    RANDOM_STATE, COLORS,
)
from src.validation import validate_parquet, quick_data_check
from src.regime_model import regime_conditional_stats, regime_persistence_metrics
from src.visualization import (
    setup_style, save_fig, plot_regime_heatmap, plot_correlation_heatmap,
)

warnings.filterwarnings('ignore', category=FutureWarning)
np.random.seed(RANDOM_STATE)

# Logging
logging.basicConfig(
    filename=str(LOGS_DIR / f'phase_05_{datetime.now():%Y%m%d_%H%M}.log'),
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
)
logger = logging.getLogger(__name__)
logger.info('Phase 5 started')

setup_style()
print('Setup complete.')

Setup complete.


In [2]:
# ------------------------------------------------------------------
# Load factor returns
# ------------------------------------------------------------------
factor_returns = pd.read_parquet(PROCESSED_DIR / 'factor_returns_full.parquet')
validate_parquet(
    factor_returns,
    expected_cols=['hml', 'umd', 'rmw', 'lowvol'],
    min_rows=100,
    date_index=True,
    label='factor_returns_full',
)

# Keep only the 4 primary factors
FACTORS = ['hml', 'umd', 'rmw', 'lowvol']
factor_returns = factor_returns[FACTORS]

quick_data_check(factor_returns, 'Factor Returns (4 primary)')

=== Factor Returns (4 primary) ===
  Shape: (232, 4)
  Index: DatetimeIndex, range: 2005-01-31 00:00:00 → 2024-04-30 00:00:00
  Duplicates: 0
  NaN total: 0 (0.0%)
  Inf total: 0
  Dtypes: {dtype('float64'): np.int64(4)}
  Sorted: True



In [3]:
# ------------------------------------------------------------------
# Load regime probabilities
# ------------------------------------------------------------------
regime_probs = pd.read_parquet(PROCESSED_DIR / 'regime_probabilities.parquet')
validate_parquet(
    regime_probs,
    expected_cols=['p_expansion', 'p_slowdown', 'p_crisis', 'regime_label'],
    min_rows=100,
    date_index=True,
    label='regime_probabilities',
)

quick_data_check(regime_probs, 'Regime Probabilities')

=== Regime Probabilities ===
  Shape: (186, 4)
  Index: DatetimeIndex, range: 2008-11-30 00:00:00 → 2024-04-30 00:00:00
  Duplicates: 0
  NaN total: 0 (0.0%)
  Inf total: 0
  Dtypes: {dtype('float64'): np.int64(3), dtype('O'): np.int64(1)}
  Sorted: True



In [4]:
# ------------------------------------------------------------------
# Align dates (inner join, month-end normalisation)
# ------------------------------------------------------------------
factor_returns.index = factor_returns.index + pd.offsets.MonthEnd(0)
regime_probs.index = regime_probs.index + pd.offsets.MonthEnd(0)

common_idx = factor_returns.index.intersection(regime_probs.index)
factor_returns = factor_returns.loc[common_idx].sort_index()
regime_probs = regime_probs.loc[common_idx].sort_index()

assert factor_returns.index.is_unique, 'Duplicate dates in factor_returns'
assert regime_probs.index.is_unique, 'Duplicate dates in regime_probs'

regime_labels = regime_probs['regime_label']

print(f'Aligned date range: {common_idx.min().date()} to {common_idx.max().date()}')
print(f'Number of months:   {len(common_idx)}')
print(f'Regime distribution:\n{regime_labels.value_counts().sort_index()}')

Aligned date range: 2008-11-30 to 2024-04-30
Number of months:   186
Regime distribution:
regime_label
Crisis       109
Expansion     53
Slowdown      24
Name: count, dtype: int64


---
## 5.2 Regime-Conditional Statistics

For each factor $f$ and regime $k$: annualised return, volatility, Sharpe, skewness,
win rate, worst/best month.

In [5]:
# Use the src function
cond_stats = regime_conditional_stats(factor_returns, regime_labels)

# ----- Add max drawdown per factor-regime -----
def max_drawdown_series(returns):
    """Maximum drawdown of a return series."""
    cum = (1 + returns).cumprod()
    running_max = cum.cummax()
    dd = (cum - running_max) / running_max
    return dd.min()

mdd_rows = []
for regime in regime_labels.unique():
    mask = regime_labels == regime
    for factor in FACTORS:
        s = factor_returns.loc[mask, factor].dropna()
        mdd = max_drawdown_series(s) if len(s) >= 3 else np.nan
        mdd_rows.append({'regime': regime, 'factor': factor, 'max_drawdown': mdd})

mdd_df = pd.DataFrame(mdd_rows)
cond_stats = cond_stats.merge(mdd_df, on=['regime', 'factor'], how='left')

print('Regime-conditional statistics:')
display_cols = ['regime', 'factor', 'ann_return', 'ann_volatility', 'sharpe',
                'skewness', 'win_rate', 'max_drawdown', 'n_months']
cond_stats[display_cols].round(4)

Regime-conditional statistics:


,regime,factor,ann_return,ann_volatility,sharpe,skewness,win_rate,max_drawdown,n_months
0,Slowdown,hml,-0.0776,0.2032,-0.3821,-0.4144,0.4167,-0.2570,24
1,Slowdown,umd,-0.2694,0.3014,-0.8936,-2.2050,0.5000,-0.5427,24
2,Slowdown,rmw,0.1106,0.0887,1.2476,0.3273,0.7083,-0.0434,24
3,Slowdown,lowvol,-0.3747,0.3807,-0.9843,-0.9394,0.4167,-0.6965,24
4,Crisis,hml,-0.0117,0.1156,-0.1010,0.7549,0.3945,-0.3389,109
5,Crisis,umd,0.0268,0.1354,0.1978,-0.9141,0.5596,-0.2249,109
6,Crisis,rmw,0.0333,0.0676,0.4922,0.4430,0.5780,-0.0790,109
7,Crisis,lowvol,-0.0718,0.1613,-0.4454,-0.1922,0.4404,-0.5640,109
8,Expansion,hml,-0.0055,0.0688,-0.0800,0.1449,0.4717,-0.2019,53
9,Expansion,umd,0.0108,0.1070,0.1011,-1.5177,0.6226,-0.1730,53


In [6]:
# Regime persistence analysis
persistence = regime_persistence_metrics(regime_labels)

if persistence:
    print('Regime duration statistics (months):')
    display(persistence['duration_stats'])
    print(f"\nRegime change frequency: {persistence['change_frequency']:.3f}")
    print(f"Total periods: {persistence['total_periods']}")

Regime duration statistics (months):


,avg_duration,median_duration,min_duration,max_duration,n_episodes
regime,,,,,
Crisis,36.333333,33.0,2,74,3
Expansion,26.500000,26.5,5,48,2
Slowdown,6.000000,6.0,5,7,4



Regime change frequency: 0.043
Total periods: 186


In [7]:
# ----- Validation gate: each regime has >= 20 months -----
regime_counts = regime_labels.value_counts()
for r, n in regime_counts.items():
    status = 'PASS' if n >= 20 else 'FAIL'
    print(f'  [{status}] {r}: {n} months (>= 20 required)')
assert (regime_counts >= 20).all(), 'At least one regime has < 20 months'

  [PASS] Crisis: 109 months (>= 20 required)
  [PASS] Expansion: 53 months (>= 20 required)
  [PASS] Slowdown: 24 months (>= 20 required)


---
## 5.3 Hypothesis Testing

### 5.3.1 Welch's t-test (pairwise between regimes) with Holm-Bonferroni correction

In [8]:
regimes_ordered = ['Expansion', 'Slowdown', 'Crisis']
regime_pairs = []
for i in range(len(regimes_ordered)):
    for j in range(i + 1, len(regimes_ordered)):
        regime_pairs.append((regimes_ordered[i], regimes_ordered[j]))

# Pairwise Welch t-tests per factor
welch_results = []
for factor in FACTORS:
    for r1, r2 in regime_pairs:
        s1 = factor_returns.loc[regime_labels == r1, factor].dropna()
        s2 = factor_returns.loc[regime_labels == r2, factor].dropna()
        t_stat, p_val = stats.ttest_ind(s1, s2, equal_var=False)
        welch_results.append({
            'factor': factor,
            'pair': f'{r1} vs {r2}',
            'mean_1': s1.mean(),
            'mean_2': s2.mean(),
            'diff': s1.mean() - s2.mean(),
            't_stat': t_stat,
            'p_raw': p_val,
            'n1': len(s1),
            'n2': len(s2),
        })

welch_df = pd.DataFrame(welch_results)

# Holm-Bonferroni correction across ALL tests
reject, pvals_corrected, _, _ = multipletests(
    welch_df['p_raw'].values, alpha=0.05, method='holm'
)
welch_df['p_holm'] = pvals_corrected
welch_df['reject_holm'] = reject

print('Welch t-test results (Holm-Bonferroni corrected):')
welch_df[['factor', 'pair', 'diff', 't_stat', 'p_raw', 'p_holm', 'reject_holm']].round(4)

Welch t-test results (Holm-Bonferroni corrected):


,factor,pair,diff,t_stat,p_raw,p_holm,reject_holm
0,hml,Expansion vs Slowdown,0.0060,0.4895,0.6287,1.0,False
1,hml,Expansion vs Crisis,0.0005,0.1225,0.9026,1.0,False
2,hml,Slowdown vs Crisis,-0.0055,-0.4435,0.6610,1.0,False
3,umd,Expansion vs Slowdown,0.0233,1.2786,0.2125,1.0,False
4,umd,Expansion vs Crisis,-0.0013,-0.2351,0.8145,1.0,False
5,umd,Slowdown vs Crisis,-0.0247,-1.3596,0.1861,1.0,False
6,rmw,Expansion vs Slowdown,-0.0089,-1.5670,0.1270,1.0,False
7,rmw,Expansion vs Crisis,-0.0025,-0.8443,0.4002,1.0,False
8,rmw,Slowdown vs Crisis,0.0064,1.1614,0.2549,1.0,False
9,lowvol,Expansion vs Slowdown,0.0151,0.6350,0.5305,1.0,False


In [9]:
# ----- Validation gate: >= 2 of 4 factors show significant regime-dependent means -----
significant_factors = set()
for factor in FACTORS:
    factor_mask = welch_df['factor'] == factor
    if welch_df.loc[factor_mask, 'reject_holm'].any():
        significant_factors.add(factor)

n_sig = len(significant_factors)
status = 'PASS' if n_sig >= 2 else 'FAIL'
print(f'\n[{status}] Factors with significant regime-dependent means (Holm p < 0.05): '
      f'{n_sig}/4 --- {significant_factors}')
if n_sig < 2:
    print(f'NOTE: Only {n_sig} factors show significant regime dependence. ')
    print('This may reflect the sample period or regime classification. Proceeding.')



[FAIL] Factors with significant regime-dependent means (Holm p < 0.05): 0/4 --- set()
NOTE: Only 0 factors show significant regime dependence. 
This may reflect the sample period or regime classification. Proceeding.


### 5.3.2 Kruskal-Wallis non-parametric test

In [10]:
kw_results = []
for factor in FACTORS:
    groups = []
    for r in regimes_ordered:
        s = factor_returns.loc[regime_labels == r, factor].dropna()
        groups.append(s.values)
    h_stat, p_val = stats.kruskal(*groups)
    kw_results.append({
        'factor': factor,
        'H_stat': h_stat,
        'p_value': p_val,
        'significant': p_val < 0.05,
    })

kw_df = pd.DataFrame(kw_results)
print('Kruskal-Wallis test (non-parametric):')
kw_df.round(4)

Kruskal-Wallis test (non-parametric):


,factor,H_stat,p_value,significant
0,hml,0.6094,0.7373,False
1,umd,1.4117,0.4937,False
2,rmw,2.5359,0.2814,False
3,lowvol,1.3259,0.5153,False


### 5.3.3 Bootstrap Sharpe ratio differences (10,000 resamples)

In [11]:
N_BOOTSTRAP = 10_000
rng = np.random.RandomState(RANDOM_STATE)


def bootstrap_sharpe(returns, n_boot=N_BOOTSTRAP, rng=rng):
    """Bootstrap distribution of annualised Sharpe ratio."""
    sharpes = np.zeros(n_boot)
    n = len(returns)
    for b in range(n_boot):
        sample = rng.choice(returns, size=n, replace=True)
        mu = sample.mean() * 12
        sigma = sample.std() * np.sqrt(12)
        sharpes[b] = mu / sigma if sigma > 0 else 0.0
    return sharpes


def bootstrap_sharpe_diff(r1, r2, n_boot=N_BOOTSTRAP, rng=rng):
    """Bootstrap the difference in Sharpe ratios between two return series."""
    diffs = np.zeros(n_boot)
    n1, n2 = len(r1), len(r2)
    for b in range(n_boot):
        s1 = rng.choice(r1, size=n1, replace=True)
        s2 = rng.choice(r2, size=n2, replace=True)
        mu1, sig1 = s1.mean() * 12, s1.std() * np.sqrt(12)
        mu2, sig2 = s2.mean() * 12, s2.std() * np.sqrt(12)
        sr1 = mu1 / sig1 if sig1 > 0 else 0.0
        sr2 = mu2 / sig2 if sig2 > 0 else 0.0
        diffs[b] = sr1 - sr2
    return diffs


# Bootstrap Sharpe per factor x regime
bootstrap_ci = []
for factor in FACTORS:
    for regime in regimes_ordered:
        r = factor_returns.loc[regime_labels == regime, factor].dropna().values
        if len(r) < 10:
            continue
        boot = bootstrap_sharpe(r)
        ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
        bootstrap_ci.append({
            'factor': factor,
            'regime': regime,
            'sharpe_mean': boot.mean(),
            'ci_2.5': ci_lo,
            'ci_97.5': ci_hi,
        })

boot_ci_df = pd.DataFrame(bootstrap_ci)
print('Bootstrap 95% CIs for Sharpe ratio (10,000 resamples):')
boot_ci_df.round(3)

Bootstrap 95% CIs for Sharpe ratio (10,000 resamples):


,factor,regime,sharpe_mean,ci_2.5,ci_97.5
0,hml,Expansion,-0.086,-1.052,0.875
1,hml,Slowdown,-0.384,-1.873,1.127
2,hml,Crisis,-0.118,-0.795,0.525
3,umd,Expansion,0.172,-0.731,1.303
4,umd,Slowdown,-0.896,-2.043,0.540
5,umd,Crisis,0.218,-0.434,0.942
6,rmw,Expansion,0.056,-0.914,0.997
7,rmw,Slowdown,1.310,-0.142,2.806
8,rmw,Crisis,0.496,-0.149,1.142
9,lowvol,Expansion,-0.996,-1.899,-0.090


---
## 5.4 Key Hypotheses

### H1: Momentum crashes in crisis-to-recovery transitions

In [12]:
# Identify crisis-to-expansion transition months:
# month t where regime_label shifted from Crisis to Expansion (or Slowdown as recovery proxy)
shifted = regime_labels.shift(1)
transition_mask = (shifted == 'Crisis') & (regime_labels.isin(['Expansion', 'Slowdown']))

# Also include the first N months after a crisis ends
RECOVERY_WINDOW = 3  # months
transition_dates = regime_labels.index[transition_mask]
recovery_dates = set()
all_dates = regime_labels.index.tolist()
for td in transition_dates:
    idx = all_dates.index(td)
    for offset in range(RECOVERY_WINDOW):
        if idx + offset < len(all_dates):
            recovery_dates.add(all_dates[idx + offset])
recovery_mask = regime_labels.index.isin(recovery_dates)

umd_transition = factor_returns.loc[recovery_mask, 'umd'].dropna()
umd_all = factor_returns['umd'].dropna()

print(f'Transition months identified: {recovery_mask.sum()}')
print(f'UMD mean during transitions:  {umd_transition.mean():.4f}')
print(f'UMD mean overall:             {umd_all.mean():.4f}')

# One-sided t-test: H1 = UMD mean during transitions < overall mean
if len(umd_transition) >= 5:
    t_stat, p_two = stats.ttest_1samp(umd_transition, popmean=umd_all.mean())
    p_one = p_two / 2 if t_stat < 0 else 1.0 - p_two / 2
    print(f'\nOne-sided t-test (UMD transition < overall):')
    print(f'  t-stat = {t_stat:.3f}, p (one-sided) = {p_one:.4f}')
    h1_result = p_one < 0.05
    print(f'  H1 supported at 5%: {h1_result}')
else:
    print('Insufficient transition months for H1 test.')
    h1_result = False

Transition months identified: 6
UMD mean during transitions:  -0.0277
UMD mean overall:             -0.0013

One-sided t-test (UMD transition < overall):
  t-stat = -1.052, p (one-sided) = 0.1706
  H1 supported at 5%: False


### H2: Value outperforms in early expansion

In [13]:
# Early expansion: first 6 months of each expansion episode
expansion_mask = regime_labels == 'Expansion'
expansion_episodes = []
in_expansion = False
ep_start = None

for i, (date, label) in enumerate(regime_labels.items()):
    if label == 'Expansion' and not in_expansion:
        in_expansion = True
        ep_start = i
    elif label != 'Expansion' and in_expansion:
        in_expansion = False
        expansion_episodes.append((ep_start, i))
if in_expansion:
    expansion_episodes.append((ep_start, len(regime_labels)))

EARLY_MONTHS = 6
early_expansion_dates = []
for start_idx, end_idx in expansion_episodes:
    ep_dates = regime_labels.index[start_idx:min(start_idx + EARLY_MONTHS, end_idx)]
    early_expansion_dates.extend(ep_dates)

early_exp_mask = regime_labels.index.isin(early_expansion_dates)
hml_early_exp = factor_returns.loc[early_exp_mask, 'hml'].dropna()
hml_rest = factor_returns.loc[~early_exp_mask, 'hml'].dropna()

print(f'Early expansion months: {len(hml_early_exp)}')
print(f'HML mean (early expansion): {hml_early_exp.mean():.4f}')
print(f'HML mean (rest):            {hml_rest.mean():.4f}')

# Two-sample t-test
if len(hml_early_exp) >= 5:
    t_stat, p_val = stats.ttest_ind(hml_early_exp, hml_rest, equal_var=False)
    print(f'\nWelch t-test (early expansion vs rest):')
    print(f'  t-stat = {t_stat:.3f}, p = {p_val:.4f}')

    # Bootstrap CI for the difference in means
    diff_boot = bootstrap_sharpe_diff(
        hml_early_exp.values, hml_rest.values, n_boot=N_BOOTSTRAP, rng=rng
    )
    ci_lo, ci_hi = np.percentile(diff_boot, [2.5, 97.5])
    print(f'  Bootstrap 95% CI for Sharpe difference: [{ci_lo:.3f}, {ci_hi:.3f}]')
    h2_result = t_stat > 0 and p_val < 0.10
    print(f'  H2 supported at 10%: {h2_result}')
else:
    print('Insufficient early-expansion months for H2 test.')
    h2_result = False

Early expansion months: 11
HML mean (early expansion): 0.0003
HML mean (rest):            -0.0017

Welch t-test (early expansion vs rest):
  t-stat = 0.207, p = 0.8396


  Bootstrap 95% CI for Sharpe difference: [-2.332, 2.815]
  H2 supported at 10%: False


### H3: Quality / LowVol dominate in crisis

In [14]:
# Pairwise bootstrap Sharpe differences in crisis: RMW and LowVol vs HML and UMD
crisis_mask = regime_labels == 'Crisis'

h3_results = []
defensive = ['rmw', 'lowvol']
cyclical  = ['hml', 'umd']

for d_factor in defensive:
    for c_factor in cyclical:
        r_d = factor_returns.loc[crisis_mask, d_factor].dropna().values
        r_c = factor_returns.loc[crisis_mask, c_factor].dropna().values
        if len(r_d) < 10 or len(r_c) < 10:
            continue
        diff_boot = bootstrap_sharpe_diff(r_d, r_c, n_boot=N_BOOTSTRAP, rng=rng)
        ci_lo, ci_hi = np.percentile(diff_boot, [2.5, 97.5])
        p_positive = (diff_boot > 0).mean()
        h3_results.append({
            'defensive': d_factor,
            'cyclical': c_factor,
            'sharpe_diff_mean': diff_boot.mean(),
            'ci_2.5': ci_lo,
            'ci_97.5': ci_hi,
            'P(diff>0)': p_positive,
        })

h3_df = pd.DataFrame(h3_results)
print('H3: Bootstrap Sharpe difference (defensive - cyclical) in Crisis:')
display(h3_df.round(3))

# H3 supported if majority of defensive-cyclical pairs have P(diff>0) > 0.75
h3_result = (h3_df['P(diff>0)'] > 0.75).mean() > 0.5 if len(h3_df) > 0 else False
print(f'\nH3 supported: {h3_result}')

H3: Bootstrap Sharpe difference (defensive - cyclical) in Crisis:


,defensive,cyclical,sharpe_diff_mean,ci_2.5,ci_97.5,P(diff>0)
0,rmw,hml,0.608,-0.299,1.528,0.905
1,rmw,umd,0.278,-0.669,1.171,0.722
2,lowvol,hml,-0.334,-1.257,0.621,0.242
3,lowvol,umd,-0.677,-1.648,0.255,0.079



H3 supported: False


In [15]:
# ----- Validation gate: UMD worst crisis Sharpe; RMW or LowVol best -----
crisis_stats = cond_stats[cond_stats['regime'] == 'Crisis'].set_index('factor')

worst_crisis_factor = crisis_stats['sharpe'].idxmin()
best_crisis_factor = crisis_stats['sharpe'].idxmax()

print(f'Worst crisis Sharpe:  {worst_crisis_factor} ({crisis_stats.loc[worst_crisis_factor, "sharpe"]:.3f})')
print(f'Best crisis Sharpe:   {best_crisis_factor}  ({crisis_stats.loc[best_crisis_factor, "sharpe"]:.3f})')

umd_worst = worst_crisis_factor == 'umd'
defensive_best = best_crisis_factor in ('rmw', 'lowvol')

print(f'\n[{"PASS" if umd_worst else "FAIL"}] UMD has worst crisis Sharpe')
print(f'[{"PASS" if defensive_best else "FAIL"}] RMW or LowVol has best crisis Sharpe')

Worst crisis Sharpe:  lowvol (-0.445)
Best crisis Sharpe:   rmw  (0.492)

[FAIL] UMD has worst crisis Sharpe
[PASS] RMW or LowVol has best crisis Sharpe


### H4: Cross-factor correlations increase in crisis (Fisher z-transform test)

In [16]:
def fisher_z(r):
    """Fisher z-transformation of a Pearson correlation."""
    r = np.clip(r, -0.9999, 0.9999)
    return 0.5 * np.log((1 + r) / (1 - r))


def fisher_z_test(r1, n1, r2, n2):
    """Test H0: rho_1 = rho_2 using Fisher z-transform.
    Returns z-statistic and two-sided p-value."""
    z1 = fisher_z(r1)
    z2 = fisher_z(r2)
    se = np.sqrt(1 / (n1 - 3) + 1 / (n2 - 3))
    z_stat = (z1 - z2) / se
    p_val = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    return z_stat, p_val

In [17]:
# Compute pairwise correlations in Expansion vs Crisis
expansion_returns = factor_returns.loc[regime_labels == 'Expansion']
crisis_returns = factor_returns.loc[regime_labels == 'Crisis']
n_exp = len(expansion_returns)
n_crisis = len(crisis_returns)

corr_exp = expansion_returns.corr()
corr_crisis = crisis_returns.corr()

fisher_results = []
factor_pairs = []
for i in range(len(FACTORS)):
    for j in range(i + 1, len(FACTORS)):
        f1, f2 = FACTORS[i], FACTORS[j]
        r_exp = corr_exp.loc[f1, f2]
        r_cri = corr_crisis.loc[f1, f2]
        z_stat, p_val = fisher_z_test(r_cri, n_crisis, r_exp, n_exp)
        factor_pairs.append(f'{f1}-{f2}')
        fisher_results.append({
            'pair': f'{f1}-{f2}',
            'corr_expansion': r_exp,
            'corr_crisis': r_cri,
            'diff': r_cri - r_exp,
            'z_stat': z_stat,
            'p_value': p_val,
        })

fisher_df = pd.DataFrame(fisher_results)

# Holm correction on Fisher z p-values
rej_f, pvals_f, _, _ = multipletests(fisher_df['p_value'].values, alpha=0.05, method='holm')
fisher_df['p_holm'] = pvals_f
fisher_df['reject_holm'] = rej_f

print('Fisher z-transform test: Crisis vs Expansion correlations')
display(fisher_df.round(4))

# Mean absolute correlation comparison
mean_abs_corr_exp = corr_exp.values[np.triu_indices_from(corr_exp.values, k=1)].__abs__().mean()
mean_abs_corr_crisis = corr_crisis.values[np.triu_indices_from(corr_crisis.values, k=1)].__abs__().mean()

h4_result = mean_abs_corr_crisis > mean_abs_corr_exp
print(f'\nMean |corr| Expansion: {mean_abs_corr_exp:.3f}')
print(f'Mean |corr| Crisis:    {mean_abs_corr_crisis:.3f}')
print(f'[{"PASS" if h4_result else "FAIL"}] Correlations higher in crisis: {h4_result}')

Fisher z-transform test: Crisis vs Expansion correlations


,pair,corr_expansion,corr_crisis,diff,z_stat,p_value,p_holm,reject_holm
0,hml-umd,-0.1619,-0.2282,-0.0663,-0.4018,0.6878,1.0000,False
1,hml-rmw,-0.4938,0.1221,0.6160,3.8695,0.0001,0.0007,True
2,hml-lowvol,-0.3033,0.0223,0.3257,1.9558,0.0505,0.2020,False
3,umd-rmw,0.4407,0.0084,-0.4323,-2.7084,0.0068,0.0338,True
4,umd-lowvol,0.4629,0.5546,0.0917,0.7230,0.4697,1.0000,False
5,rmw-lowvol,0.5381,0.4157,-0.1225,-0.9273,0.3538,1.0000,False



Mean |corr| Expansion: 0.400
Mean |corr| Crisis:    0.225
[FAIL] Correlations higher in crisis: False


---
## 5.5 Correlation Analysis by Regime

In [18]:
# Compute correlation matrix per regime
corr_by_regime = {}
for regime in regimes_ordered:
    mask = regime_labels == regime
    r = factor_returns.loc[mask]
    corr_by_regime[regime] = r.corr()

# Print side-by-side
for regime, corr_mat in corr_by_regime.items():
    print(f'\n--- {regime} (n={int((regime_labels == regime).sum())}) ---')
    display(corr_mat.round(3))


--- Expansion (n=53) ---


,hml,umd,rmw,lowvol
hml,1.000,-0.162,-0.494,-0.303
umd,-0.162,1.000,0.441,0.463
rmw,-0.494,0.441,1.000,0.538
lowvol,-0.303,0.463,0.538,1.000



--- Slowdown (n=24) ---


,hml,umd,rmw,lowvol
hml,1.000,-0.541,0.025,-0.600
umd,-0.541,1.000,0.024,0.818
rmw,0.025,0.024,1.000,0.350
lowvol,-0.600,0.818,0.350,1.000



--- Crisis (n=109) ---


,hml,umd,rmw,lowvol
hml,1.000,-0.228,0.122,0.022
umd,-0.228,1.000,0.008,0.555
rmw,0.122,0.008,1.000,0.416
lowvol,0.022,0.555,0.416,1.000


---
## 5.6 Visualizations

### 5.6.1 Regime-Conditional Sharpe Heatmap

In [19]:
fig = plot_regime_heatmap(cond_stats, metric='sharpe')
save_fig(fig, 'regime_conditional_sharpe_heatmap')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/regime_conditional_sharpe_heatmap.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/regime_conditional_sharpe_heatmap.png')

In [20]:
# Also plot annualised return heatmap
fig = plot_regime_heatmap(cond_stats, metric='ann_return')
save_fig(fig, 'regime_conditional_return_heatmap')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/regime_conditional_return_heatmap.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/regime_conditional_return_heatmap.png')

### 5.6.2 Factor Cumulative Returns Colored by Regime

In [21]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)

regime_color_map = {
    'Expansion': COLORS['expansion'],
    'Slowdown':  COLORS['slowdown'],
    'Crisis':    COLORS['crisis'],
}
factor_color_map = {
    'hml': COLORS['hml'],
    'umd': COLORS['umd'],
    'rmw': COLORS['rmw'],
    'lowvol': COLORS['lowvol'],
}

for ax, factor in zip(axes.flat, FACTORS):
    cum_ret = (1 + factor_returns[factor]).cumprod()
    ax.plot(cum_ret.index, cum_ret.values, color=factor_color_map[factor],
            linewidth=1.2, zorder=3)

    # Regime shading
    for i in range(len(regime_labels) - 1):
        label = regime_labels.iloc[i]
        color = regime_color_map.get(label, '#cccccc')
        ax.axvspan(regime_labels.index[i], regime_labels.index[i + 1],
                   alpha=0.12, color=color, linewidth=0)

    ax.set_title(f'{factor.upper()} Cumulative Return', fontweight='bold')
    ax.set_ylabel('Cumulative Return (base=1)')
    ax.grid(True, alpha=0.3)

# Add shared legend for regimes
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=COLORS['expansion'], alpha=0.3, label='Expansion'),
    Patch(facecolor=COLORS['slowdown'],  alpha=0.3, label='Slowdown'),
    Patch(facecolor=COLORS['crisis'],    alpha=0.3, label='Crisis'),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=3,
           fontsize=11, bbox_to_anchor=(0.5, 1.02))

plt.tight_layout(rect=[0, 0, 1, 0.97])
save_fig(fig, 'factor_cumulative_returns_by_regime')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/factor_cumulative_returns_by_regime.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/factor_cumulative_returns_by_regime.png')

### 5.6.3 Box Plots of Factor Returns by Regime

In [22]:
# Melt data for seaborn
plot_data = factor_returns.copy()
plot_data['regime'] = regime_labels
melted = plot_data.melt(id_vars='regime', var_name='factor', value_name='return')

fig, axes = plt.subplots(1, 4, figsize=(18, 6), sharey=True)

palette = {
    'Expansion': COLORS['expansion'],
    'Slowdown':  COLORS['slowdown'],
    'Crisis':    COLORS['crisis'],
}

for ax, factor in zip(axes, FACTORS):
    subset = melted[melted['factor'] == factor]
    sns.boxplot(
        data=subset, x='regime', y='return', order=regimes_ordered,
        palette=palette, ax=ax, showfliers=True, fliersize=3,
    )
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--', alpha=0.6)
    ax.set_title(f'{factor.upper()}', fontweight='bold', fontsize=13)
    ax.set_xlabel('')
    if ax == axes[0]:
        ax.set_ylabel('Monthly Return')
    else:
        ax.set_ylabel('')

fig.suptitle('Factor Return Distributions by Regime', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'factor_boxplots_by_regime')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/factor_boxplots_by_regime.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/factor_boxplots_by_regime.png')

### 5.6.4 Correlation Heatmaps per Regime

In [23]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, regime in zip(axes, regimes_ordered):
    corr = corr_by_regime[regime]
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(
        corr, mask=mask, annot=True, fmt='.2f',
        cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        square=True, ax=ax,
        cbar_kws={'shrink': 0.8},
    )
    n = int((regime_labels == regime).sum())
    ax.set_title(f'{regime} (n={n})', fontweight='bold', fontsize=13)

fig.suptitle('Factor Correlation Matrices by Regime', fontsize=15,
             fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, 'correlation_heatmaps_by_regime')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/correlation_heatmaps_by_regime.png


PosixPath('/Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/figures/correlation_heatmaps_by_regime.png')

---
## 5.7 Summary of Hypothesis Test Results

In [24]:
print('=' * 70)
print('HYPOTHESIS TEST SUMMARY')
print('=' * 70)

hypotheses = [
    ('H1', 'Momentum crashes in crisis-to-recovery transitions', h1_result),
    ('H2', 'Value outperforms in early expansion', h2_result),
    ('H3', 'Quality/LowVol dominate in crisis', h3_result),
    ('H4', 'Cross-factor correlations increase in crisis', h4_result),
]

for hid, desc, result in hypotheses:
    status = 'SUPPORTED' if result else 'NOT SUPPORTED'
    print(f'  {hid}: {desc}')
    print(f'       => {status}')
    print()

n_supported = sum(r for _, _, r in hypotheses)
print(f'Hypotheses supported: {n_supported} / {len(hypotheses)}')

logger.info(f'Hypothesis results: {n_supported}/4 supported')
for hid, desc, result in hypotheses:
    logger.info(f'  {hid} ({"supported" if result else "not supported"}): {desc}')

HYPOTHESIS TEST SUMMARY
  H1: Momentum crashes in crisis-to-recovery transitions
       => NOT SUPPORTED

  H2: Value outperforms in early expansion
       => NOT SUPPORTED

  H3: Quality/LowVol dominate in crisis
       => NOT SUPPORTED

  H4: Cross-factor correlations increase in crisis
       => NOT SUPPORTED

Hypotheses supported: 0 / 4


---
## 5.8 Validation Gates Summary

In [25]:
print('=' * 70)
print('VALIDATION GATES')
print('=' * 70)

# Gate 1: >= 2 of 4 factors show significant regime-dependent means
gate1 = n_sig >= 2
print(f'  [1] >= 2/4 factors with significant regime means (Holm p<0.05): '
      f'{"PASS" if gate1 else "FAIL"} ({n_sig}/4)')

# Gate 2: UMD worst crisis; RMW/LowVol best
gate2 = umd_worst and defensive_best
print(f'  [2] UMD worst crisis Sharpe, RMW/LowVol best: '
      f'{"PASS" if gate2 else "FAIL"} (worst={worst_crisis_factor}, best={best_crisis_factor})')

# Gate 3: Correlations higher in crisis than expansion
gate3 = h4_result
print(f'  [3] Correlations higher in crisis than expansion: '
      f'{"PASS" if gate3 else "FAIL"}')

# Gate 4: Each regime >= 20 months
gate4 = (regime_counts >= 20).all()
print(f'  [4] Each regime >= 20 months: {"PASS" if gate4 else "FAIL"}')

# Gate 5: Bootstrap CIs reported
gate5 = len(boot_ci_df) > 0
print(f'  [5] Bootstrap CIs reported: {"PASS" if gate5 else "FAIL"}')

all_pass = gate1 and gate2 and gate3 and gate4 and gate5
print(f'\n  OVERALL: {"ALL GATES PASSED" if all_pass else "SOME GATES FAILED"}')

logger.info(f'Validation gates: {"ALL PASSED" if all_pass else "SOME FAILED"}')

VALIDATION GATES
  [1] >= 2/4 factors with significant regime means (Holm p<0.05): FAIL (0/4)
  [2] UMD worst crisis Sharpe, RMW/LowVol best: FAIL (worst=lowvol, best=rmw)
  [3] Correlations higher in crisis than expansion: FAIL
  [4] Each regime >= 20 months: PASS
  [5] Bootstrap CIs reported: PASS

  OVERALL: SOME GATES FAILED


---
## 5.9 Export Outputs

In [26]:
# Save regime-conditional stats table
output_path = TABLES_DIR / 'regime_conditional_stats.csv'
cond_stats.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
print(f'  Rows: {len(cond_stats)}')

# Save Welch t-test results
welch_path = TABLES_DIR / 'welch_ttest_regime_pairwise.csv'
welch_df.to_csv(welch_path, index=False)
print(f'Saved: {welch_path}')

# Save bootstrap CIs
boot_path = TABLES_DIR / 'bootstrap_sharpe_ci.csv'
boot_ci_df.to_csv(boot_path, index=False)
print(f'Saved: {boot_path}')

# Save Fisher z-test results
fisher_path = TABLES_DIR / 'fisher_z_correlation_test.csv'
fisher_df.to_csv(fisher_path, index=False)
print(f'Saved: {fisher_path}')

# Save hypothesis summary
hyp_summary = pd.DataFrame(hypotheses, columns=['hypothesis', 'description', 'supported'])
hyp_path = TABLES_DIR / 'hypothesis_test_summary.csv'
hyp_summary.to_csv(hyp_path, index=False)
print(f'Saved: {hyp_path}')

logger.info('Phase 5 outputs exported successfully.')
logger.info('Phase 5 complete.')
print('\nPhase 5 complete.')

Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/tables/regime_conditional_stats.csv
  Rows: 12
Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/tables/welch_ttest_regime_pairwise.csv
Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/tables/bootstrap_sharpe_ci.csv
Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/tables/fisher_z_correlation_test.csv
Saved: /Users/laurentnguyen/Documents/Personal Project/Finance/factor-timing-engine/outputs/tables/hypothesis_test_summary.csv

Phase 5 complete.
